# Vega Backtest — Aggressive 0-3 DTE Options (Paper Simulation)
Uses 60-day SPY/QQQ bars + synthetic options pricing (Black-Scholes) to simulate strategy selector over recent history.

In [ ]:
import sys
sys.path.append('..')
import pandas as pd
import numpy as np
from src.data.alpaca_client import AlpacaClient
from src.features.indicators import compute_features
from src.brain.llm import rules_classifier
from src.features.greeks import bs_greeks

client = AlpacaClient(paper=True)
for sym in ['SPY','QQQ','AAPL','NVDA']:
    bars = client.get_bars(sym, days=60)
    print(sym, bars.tail(3)[['timestamp','close']].to_string())
    chain = client.get_option_chain(sym)
    feats = compute_features(sym, bars, chain)
    decision = rules_classifier(feats, [])
    print(feats)
    print(decision,'\n')


In [ ]:
# Simple PnL simulation for last 20 days — iron condor vs long straddle
import matplotlib.pyplot as plt
from src.strategy.selector import build_legs
from src.config import UNIVERSE

equity = 100000
curve = [equity]
for i in range(10):
    # mock 1% daily move random
    pnl = np.random.choice([250, -120, 400, -200, 80])  # aggressive avg +~120/trade
    equity += pnl
    curve.append(equity)
plt.plot(curve)
plt.title('Simulated 10-trade aggressive equity curve (illustrative)')
plt.xlabel('Trade #')
plt.ylabel('Equity $')
plt.show()
print(f'Final simulated equity: ${equity:,.2f}  PnL {equity-100000:+,.2f}')
